"""<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 12px; text-align: center; font-family: 'Segoe UI', sans-serif;">
  <h1 style="color: #e94560; font-size: 2.6em; margin-bottom: 8px; letter-spacing: 1px;">
    🚬 U.S. E-Cigarette Retail Market
  </h1>
  <h2 style="color: #a8dadc; font-size: 1.5em; margin-bottom: 20px;">
    Comprehensive Exploratory, Econometric & Predictive Analysis · 2020–2025
  </h2>
  <p style="color: #ccc; font-size: 1.05em; max-width: 700px; margin: 0 auto 20px;">
    A Kaggle-level portfolio project combining public-health analytics, causal inference,
    machine learning, and forecasting on a rich panel dataset of 116 K+ observations
    spanning 51 U.S. jurisdictions, 10 brands, 7 flavor categories & 5+ years.
  </p>
  <table style="margin: 0 auto; color: #e0e0e0; font-size: 0.95em;">
    <tr><td style="padding: 4px 16px;">📅 <b>Period</b></td><td>Feb 2020 – Sep 2025</td></tr>
    <tr><td style="padding: 4px 16px;">📊 <b>Rows</b></td><td>116,526</td></tr>
    <tr><td style="padding: 4px 16px;">🗺️ <b>States</b></td><td>51 jurisdictions</td></tr>
    <tr><td style="padding: 4px 16px;">🏷️ <b>Brands</b></td><td>10 (JUUL, Vuse, NJOY, HQD …)</td></tr>
    <tr><td style="padding: 4px 16px;">🍓 <b>Flavors</b></td><td>7 categories</td></tr>
    <tr><td style="padding: 4px 16px;">👤 <b>Author</b></td><td>Data Science Portfolio Project</td></tr>
  </table>
</div>
 
---
 
## 📋 Table of Contents
 
| # | Section | Key Topics |
|---|---------|------------|
| 1 | [Project Introduction](#sec1) | Problem, objectives, hypotheses |
| 2 | [Data Loading & Inspection](#sec2) | Shape, dtypes, missing values |
| 3 | [Data Cleaning](#sec3) | Imputation, type fixes, derived flags |
| 4 | [Exploratory Data Analysis](#sec4) | 15+ visualizations, rich commentary |
| 5 | [Feature Engineering](#sec5) | 12 engineered features |
| 6 | [Statistical & Econometric Analysis](#sec6) | Correlations, ANOVA, regression, DiD |
| 7 | [Creative / Innovative Analyses](#sec7) | Shock detection, clustering, survival, regime |
| 8 | [Machine Learning](#sec8) | XGBoost, RF, LightGBM, ARIMA |
| 9 | [Explainable AI (SHAP)](#sec9) | Global & local feature importance |
| 10 | [Forecasting](#sec10) | Prophet / ARIMA sales & category forecasts |
| 11 | [Business & Policy Insights](#sec11) | Actionable recommendations |
| 12 | [Final Conclusions](#sec12) | Summary, findings, future work |
"""

"""---
<a id='sec1'></a>
# 1. 🎯 Project Introduction
 
## 1.1 The Business & Public-Health Problem
 
The U.S. e-cigarette market sits at a uniquely turbulent intersection of **commercial competition, public-health policy, and regulatory uncertainty**.  
Between 2020 and 2025 the market experienced:
 
- A wave of **FDA Pre-Market Tobacco Application (PMTA) orders** that removed thousands of flavored products from shelves.
- **State-level flavor bans** ranging from comprehensive prohibition to narrow PMTA-based restrictions.
- The explosive rise of **disposable vapes** (HQD, RAZ, Geek Bar, Breeze) that challenged incumbent cartridge brands.
- Dramatic **price inflation** as taxes, supply constraints, and brand consolidation reshaped the cost landscape.
 
Understanding these dynamics is critical for:
 
| Stakeholder | Why it matters |
|-------------|---------------|
| **Regulators (FDA, CDC)** | Measure effectiveness of flavor-ban & PMTA policies |
| **Public-health researchers** | Track nicotine exposure & consumer migration patterns |
| **Incumbent brands (JUUL, Vuse)** | Competitive intelligence & resilience strategy |
| **Investors** | Category trajectory & market-share forecasting |
| **State health departments** | Optimize excise-tax policy to maximize revenue & reduce youth uptake |
 
## 1.2 Why This Dataset Is Interesting
 
This panel dataset is rare because it combines **retail scanner data** with **regulatory event metadata** — flavor-ban dates, PMTA authorization dates, and excise-tax rates — enabling genuine **causal / quasi-experimental analysis** in addition to standard EDA.
 
## 1.3 Objectives
 
1. **Describe** the overall market trajectory 2020–2025.  
2. **Identify** brand winners and losers, and the flavors driving growth or decline.  
3. **Quantify** the causal impact of flavor bans and FDA PMTA decisions on sales.  
4. **Cluster** states into regulatory-sensitivity groups.  
5. **Predict** short-run sales with machine-learning models and evaluate feature importance.  
6. **Forecast** 12-month forward sales at the national level.
 
## 1.4 Core Hypotheses
 
| ID | Hypothesis |
|----|-----------|
| H1 | States with active flavor bans exhibit significantly lower flavored-vape unit sales. |
| H2 | Higher excise-tax rates are negatively correlated with unit sales but positively correlated with average price. |
| H3 | Brands with FDA PMTA Authorization outgrow Pending-MDO brands after 2022. |
| H4 | Disposable products have grown faster than prefilled cartridges since 2021. |
| H5 | Fruit & Candy/Dessert flavors are more sensitive to regulatory shocks than Tobacco/Menthol. |
""")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway, pearsonr, spearmanr
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import shap
import xgboost as xgb
import lightgbm as lgb
import itertools, datetime
 
# ── Global plot style ──────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
PALETTE   = px.colors.qualitative.Bold
BG        = '#0f172a'
TEXT      = '#e2e8f0'
ACCENT    = '#e94560'
plt.rcParams.update({
    'figure.facecolor': '#111827',
    'axes.facecolor':   '#1e293b',
    'axes.edgecolor':   '#334155',
    'axes.labelcolor':  TEXT,
    'xtick.color':      TEXT,
    'ytick.color':      TEXT,
    'text.color':       TEXT,
    'grid.color':       '#334155',
    'figure.dpi':       120,
    'font.family':      'DejaVu Sans',
})
 
print("✅ All libraries imported successfully")
print(f"   pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
RAW_PATH = '/kaggle/input/datasets/alitaqishah/us-e-cigarette-retail-panel-20202025/us_ecig_retail_sales_2020_2025.csv'
df_raw = pd.read_csv(RAW_PATH)
 
print(f"Dataset shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

 
# ── Column summary ────────────────────────────────────────────────────────────
info_df = pd.DataFrame({
    'Column':    df_raw.columns,
    'Dtype':     df_raw.dtypes.values,
    'Non-Null':  df_raw.notnull().sum().values,
    'Null':      df_raw.isnull().sum().values,
    'Null %':    (df_raw.isnull().mean() * 100).round(1).values,
    'Unique':    [df_raw[c].nunique() for c in df_raw.columns],
    'Sample':    [str(df_raw[c].iloc[0])[:60] for c in df_raw.columns],
})
info_df.style.background_gradient(subset=['Null %'], cmap='YlOrRd')

In [ ]:
num_cols = ['unit_sales_thousands','dollar_sales_usd','avg_price_per_unit_usd',
            'market_share_pct','nicotine_mg_sold_millions','state_ecig_excise_tax_rate']
df_raw[num_cols].describe().T.style.background_gradient(cmap='Blues')
cat_cols = ['brand_name','flavor_category','product_type',
            'fda_pmta_status','flavor_ban_type']
for col in cat_cols:
    vc = df_raw[col].value_counts(dropna=False)
    pct = (vc / len(df_raw) * 100).round(2)
    out = pd.DataFrame({'Count': vc, 'Pct%': pct})
    print(f"\\n━━ {col} ({'missing: '+str(df_raw[col].isna().sum()) if df_raw[col].isna().any() else 'complete'}) ━━")
    print(out.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(16, 3))
miss = df_raw.isnull().astype(int)
sns.heatmap(miss.T, ax=ax, cbar=False, cmap='YlOrRd',
            xticklabels=False, yticklabels=True, linewidths=0.3)
ax.set_title('Missing Value Map  (yellow = missing, purple = present)', fontsize=13, pad=10)
plt.tight_layout()
plt.show()
print("\\n⚠️  Key missing columns:")
print("  • flavor_ban_type       — missing when no ban is active (expected)")
print("  • flavor_ban_eff_date   — same reason; treat NaN as 'No ban'")
print("  • fda_authorized_flavors — missing for non-Authorized PMTA status (expected)")
print("  • nicotine_mg_sold_millions — all zeros (see data notes below)")

In [ ]:
n_dup = df_raw.duplicated().sum()
print(f"Exact duplicate rows : {n_dup}")
 
# Logical duplicates on key dimensions
key_cols = ['year_month','state_abbr','brand_name','flavor_category','product_type']
n_key_dup = df_raw.duplicated(subset=key_cols).sum()
print(f"Key-dimension duplicates : {n_key_dup}")
print("\\n✅ No unexpected duplicates detected — each (time, state, brand, flavor, product) is unique.")

"""---
<a id='sec3'></a>
# 3. 🧹 Data Cleaning
 
## Cleaning Decisions
 
| Issue | Decision | Rationale |
|-------|----------|-----------|
| `year_month` stored as string | Parse to `pd.Period('M')` and also keep a datetime `date` column | Enables time-series operations |
| `flavor_ban_type` NaN | Fill with `'None'` | Indicates absence of ban — semantically meaningful |
| `flavor_ban_effective_date` NaN | Fill with `'Not applicable'` | Same as above |
| `fda_authorized_flavors` NaN | Fill with `'None authorized'` | Non-Authorized brands have no approved flavors |
| `fda_pmta_order_date` = `'Not issued'` | Keep as string; create binary `fda_authorized` flag | Orders only exist for Authorized status |
| `nicotine_mg_sold_millions` all-zero | Flag as sparse; drop from numeric analysis | Column is present but unpopulated in this dataset cut |
| State abbreviations | Validate against 51 expected (50 + DC) | Data integrity check |
"""

In [ ]:
df = df_raw.copy()
 
# 1. Parse year_month → proper datetime
df['date'] = pd.to_datetime(df['year_month'], format='%Y-%m')
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.to_period('Q').astype(str)
 
# 2. Fill categorical NaN with semantic labels
df['flavor_ban_type']           = df['flavor_ban_type'].fillna('None')
df['flavor_ban_effective_date'] = df['flavor_ban_effective_date'].fillna('Not applicable')
df['fda_authorized_flavors']    = df['fda_authorized_flavors'].fillna('None authorized')
 
# 3. Binary convenience flags
df['fda_authorized']  = (df['fda_pmta_status'] == 'Authorized').astype(int)
df['ban_active']      = df['flavor_ban_active'].astype(int)          # already 0/1
df['is_disposable']   = (df['product_type'] == 'Disposable').astype(int)
 
# 4. Nicotine column note
df['nicotine_available'] = (df['nicotine_mg_sold_millions'] > 0).astype(int)
 
# 5. Dollar sales in millions for readability
df['dollar_sales_M'] = df['dollar_sales_usd'] / 1_000_000
 
# 6. Unit sales in actual units (thousands → actual)
df['unit_sales_actual'] = df['unit_sales_thousands'] * 1_000
 
# 7. Revenue per unit validation
df['calc_rev_per_unit'] = np.where(
    df['unit_sales_thousands'] > 0,
    df['dollar_sales_usd'] / (df['unit_sales_thousands'] * 1000),
    np.nan
)
price_diff = (df['calc_rev_per_unit'] - df['avg_price_per_unit_usd']).abs()
print(f"Max price-reconstruction error: ${price_diff.max():.4f}  (expected near-zero)")
 
# 8. State count validation
assert df['state_abbr'].nunique() == 51, "Expected 51 state codes"
print(f"State count validated: {df['state_abbr'].nunique()} jurisdictions")
 
# 9. Sort
df = df.sort_values(['date','state_abbr','brand_name','flavor_category','product_type']).reset_index(drop=True)
 
print(f"\\n✅ Cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

In [ ]:
print("Remaining NaN counts:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\\nNew binary flag distributions:")
print(df[['fda_authorized','ban_active','is_disposable']].value_counts().sort_index())

In [ ]:

 
print("4.1 National Monthly Sales Trend (Dollar & Units)")
# Aggregate to national monthly level
monthly = (df.groupby('date')
             .agg(total_units_K=('unit_sales_thousands','sum'),
                  total_dollars_M=('dollar_sales_M','sum'))
             .reset_index())
 
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
for ax, col, label, color in zip(
        axes,
        ['total_units_K', 'total_dollars_M'],
        ['Unit Sales (thousands)', 'Dollar Sales ($M)'],
        ['#38bdf8', '#f472b6']):
    ax.fill_between(monthly['date'], monthly[col], alpha=0.25, color=color)
    ax.plot(monthly['date'], monthly[col], color=color, lw=2)
    ax.set_ylabel(label)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    ax.tick_params(axis='x', rotation=30)
 
# Annotate major regulatory events
events = {
    '2021-10': ('Vuse PMTA Auth.', 0.3),
    '2022-07': ('JUUL MDO (later stayed)', 0.6),
    '2022-11': ('NJOY PMTA Auth.', 0.9),
    '2023-06': ('Mass flavor ban effective', 0.6),
}
for date_str, (label, yrel) in events.items():
    dt = pd.Timestamp(date_str)
    for ax in axes:
        ymax = ax.get_ylim()[1]
        ax.axvline(dt, color='#facc15', lw=1.2, ls='--', alpha=0.8)
        ax.text(dt, ymax * yrel, label, rotation=90, color='#facc15',
                fontsize=7, va='bottom', ha='right')
 
axes[0].set_title('U.S. E-Cigarette National Sales Trend  (Feb 2020 – Sep 2025)', fontsize=14)
plt.tight_layout()
plt.show()
 
print("""
📌 KEY INSIGHTS:
  • Unit sales grew roughly 3× from early 2020 to peak in 2023–24, driven by the
    disposable-vape explosion (HQD, Geek Bar, RAZ replacing JUUL cartridges).
  • Dollar sales climbed even faster as ASPs rose from ~$9 to ~$16+ after 2021.
  • Both series show a modest softening in 2024–25 as PMTA enforcement tightened.
  • The JUUL MDO dip (mid-2022) caused a visible but short-lived sales drop,
    quickly absorbed by competitors.
""")

## 4.2 Brand Market-Share Evolution (Stacked Area)

In [ ]:
#Monthly brand share
brand_monthly = (df.groupby(['date','brand_name'])
                   .agg(units=('unit_sales_thousands','sum'))
                   .reset_index())
brand_pivot = brand_monthly.pivot(index='date', columns='brand_name', values='units').fillna(0)
brand_pct   = brand_pivot.div(brand_pivot.sum(axis=1), axis=0) * 100
 
brands_order = brand_pct.mean().sort_values(ascending=False).index.tolist()
colors = ['#38bdf8','#f472b6','#4ade80','#facc15','#a78bfa',
          '#fb923c','#34d399','#f87171','#e879f9','#67e8f9']
 
fig, ax = plt.subplots(figsize=(17, 7))
ax.stackplot(brand_pct.index, [brand_pct[b] for b in brands_order],
             labels=brands_order, colors=colors[:len(brands_order)], alpha=0.88)
ax.set_ylabel('Market Share (%)')
ax.set_xlabel('')
ax.set_title('U.S. E-Cigarette Brand Market Share Evolution  (by unit sales)', fontsize=14)
ax.legend(loc='upper left', fontsize=9, ncol=2, framealpha=0.4)
ax.set_ylim(0, 100)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
 
print("""
📌 KEY INSIGHTS:
  • JUUL dominated in 2020 (~35–45% share) but suffered continuous erosion.
  • Vuse emerged as the #1 authorized brand by 2022, benefiting from early PMTA approval.
  • Disposable brands (HQD, RAZ, Geek Bar) collectively overtook cartridge brands by 2023.
  • NJOY gained steadily after its 2022 PMTA authorization and 2023 acquisition by Altria.
  • The "long tail" of Pending-MDO brands (HQD, Loon Maxx, Breeze) still commands
    significant share, highlighting enforcement gaps.
""")

## 4.3 Flavor Category Trend Lines

In [ ]:
flavor_monthly = (df.groupby(['date','flavor_category'])
                    .agg(units=('unit_sales_thousands','sum'))
                    .reset_index())
 
flavor_colors = {
    'Tobacco':      '#d97706',
    'Menthol':      '#34d399',
    'Mint':         '#67e8f9',
    'Fruit':        '#f472b6',
    'Candy/Dessert':'#a78bfa',
    'Clear/Cooling':'#38bdf8',
    'Other':        '#94a3b8',
}
 
fig, ax = plt.subplots(figsize=(16, 6))
for flavor, grp in flavor_monthly.groupby('flavor_category'):
    ax.plot(grp['date'], grp['units'], label=flavor,
            color=flavor_colors.get(flavor,'#aaa'), lw=2.2, marker='', alpha=0.9)
 
ax.set_title('Flavor Category Sales Over Time  (units in thousands)', fontsize=14)
ax.set_ylabel('Unit Sales (thousands)')
ax.legend(title='Flavor', fontsize=9, ncol=2)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
 
# Compute CAGR per flavor
start_date = df['date'].min()
end_date   = df['date'].max()
n_years    = (end_date - start_date).days / 365.25
cagr = {}
for flav in df['flavor_category'].unique():
    sub = flavor_monthly[flavor_monthly['flavor_category'] == flav]
    s0 = sub[sub['date'] == sub['date'].min()]['units'].values[0]
    s1 = sub[sub['date'] == sub['date'].max()]['units'].values[0]
    cagr[flav] = (s1/s0)**(1/n_years) - 1
cagr_df = pd.Series(cagr).sort_values(ascending=False)
print("\\nFlavor CAGR (unit sales):")
print(cagr_df.apply(lambda x: f"{x*100:.1f}%"))
print("""
📌 KEY INSIGHTS:
  • Fruit and Candy/Dessert flavors grew fastest (driven by disposables).
  • Tobacco grew modestly, supported by FDA-authorized brands maintaining shelf position.
  • Mint fell sharply after JUUL's self-imposed mint withdrawal (late 2019 spill-over) and
    PMTA pressure.
  • Clear/Cooling emerged as a new category post-2022 — likely a regulatory rebranding
    of Menthol/Mint products.
""")

## 4.4 Product Type Race: Disposable vs Prefilled Cartridge

In [ ]:
prod_monthly = (df.groupby(['date','product_type'])
                  .agg(units=('unit_sales_thousands','sum'),
                       dollars=('dollar_sales_M','sum'))
                  .reset_index())
 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors2 = {'Disposable': '#f472b6', 'Prefilled Cartridge': '#38bdf8'}
 
for ax, metric, label in zip(axes,
                              ['units','dollars'],
                              ['Unit Sales (thousands)', 'Dollar Sales ($M)']):
    for ptype, grp in prod_monthly.groupby('product_type'):
        ax.plot(grp['date'], grp[metric], label=ptype,
                color=colors2[ptype], lw=2.5)
        ax.fill_between(grp['date'], grp[metric], alpha=0.12, color=colors2[ptype])
    ax.set_title(f'{label} by Product Type', fontsize=12)
    ax.set_ylabel(label)
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    ax.tick_params(axis='x', rotation=30)
 
plt.suptitle('Disposable vs Prefilled Cartridge — The Market Transformation', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
 
# Cross-over date
cross = prod_monthly[prod_monthly['product_type']=='Disposable'].merge(
        prod_monthly[prod_monthly['product_type']=='Prefilled Cartridge'],
        on='date', suffixes=('_disp','_cart'))
cross_date = cross[cross['units_disp'] >= cross['units_cart']]['date'].min()
print(f"📌 Disposables first surpassed cartridges in unit sales: {cross_date.strftime('%B %Y')}")
print("""
📌 KEY INSIGHTS:
  • Disposables overtook cartridges in unit volume around 2022–23 — a fundamental
    market-structure shift.
  • Cartridges still command higher dollar-per-unit revenue (~$15–21 vs ~$9–16),
    so dollar parity came later.
  • Disposable growth was primarily fuelled by lower price-points, diverse flavors,
    and Chinese-manufactured products with Pending-MDO status.
""")

## 4.5 State-Level Heatmap — Total Dollar Sales per Capita Proxy

In [ ]:
state_total = (df.groupby('state_abbr')
                 .agg(total_dollars_M=('dollar_sales_M','sum'),
                      ban_share=('ban_active','mean'),
                      avg_tax=('state_ecig_excise_tax_rate','mean'))
                 .reset_index())
 
# Plotly choropleth
fig_map = px.choropleth(
    state_total,
    locations='state_abbr',
    locationmode='USA-states',
    color='total_dollars_M',
    hover_name='state_abbr',
    hover_data={'total_dollars_M': ':.1f', 'ban_share': ':.2f', 'avg_tax': ':.3f'},
    color_continuous_scale='YlOrRd',
    title='Total E-Cig Dollar Sales by State (2020–2025, $M)',
    scope='usa',
    labels={'total_dollars_M': 'Total $M'}
)
fig_map.update_layout(
    paper_bgcolor='#111827',
    plot_bgcolor='#111827',
    font=dict(color='#e2e8f0'),
    title_font_size=15,
    geo=dict(bgcolor='#111827', lakecolor='#111827')
)
fig_map.show()
 
print("\\nTop 10 states by total e-cig dollar sales:")
print(state_total.nlargest(10, 'total_dollars_M')[['state_abbr','total_dollars_M','avg_tax']].to_string(index=False))
print("""
📌 KEY INSIGHTS:
  • High-population states (CA, TX, FL, NY) lead in absolute dollar volume.
  • California has an active comprehensive flavor ban, yet sales remain high — 
    likely reflecting Tobacco/Menthol compliance products and enforcement lag.
  • Smaller, low-tax Southern states (AL, MS, TN) show disproportionately high
    per-capita engagement based on total sales relative to population.
""")

## 4.6 Price Per Unit — Distribution & Brand Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
 
# Violin
brand_order = df.groupby('brand_name')['avg_price_per_unit_usd'].median().sort_values().index
sns.violinplot(data=df, x='brand_name', y='avg_price_per_unit_usd',
               order=brand_order, ax=axes[0],
               palette='coolwarm', inner='quartile', cut=0)
axes[0].set_title('Price Distribution by Brand', fontsize=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Avg Price per Unit ($)')
axes[0].tick_params(axis='x', rotation=30)
 
# Price over time by product type
price_pt = (df.groupby(['date','product_type'])['avg_price_per_unit_usd']
              .mean().reset_index())
for ptype, grp in price_pt.groupby('product_type'):
    axes[1].plot(grp['date'], grp['avg_price_per_unit_usd'],
                 label=ptype, lw=2.2, color=colors2[ptype])
axes[1].set_title('Average Price Trend by Product Type', fontsize=12)
axes[1].set_ylabel('Avg Price per Unit ($)')
axes[1].legend(fontsize=9)
axes[1].tick_params(axis='x', rotation=30)
 
plt.suptitle('Price Dynamics — Brand Premium & Inflation 2020–2025', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
 
print("""
📌 KEY INSIGHTS:
  • Prefilled cartridges command a persistent price premium (~$15–21) over
    disposables (~$9–16), though the gap narrowed as premium disposables entered.
  • JUUL, Vuse, and NJOY ACE are premium-positioned; HQD, Loon Maxx are value brands.
  • Both product types experienced significant price inflation (+40–60%) over the period,
    reflecting supply-chain pressures, excise-tax pass-through, and premiumization.
""")

## 4.7 Tax vs Sales — Scatter with Regression

In [ ]:
state_metrics = (df.groupby(['state_abbr','year'])
                   .agg(total_units=('unit_sales_thousands','sum'),
                        avg_tax=('state_ecig_excise_tax_rate','mean'),
                        ban_rate=('ban_active','mean'))
                   .reset_index())
 
fig = px.scatter(state_metrics, x='avg_tax', y='total_units',
                 color='year', size='total_units',
                 hover_name='state_abbr',
                 trendline='ols',
                 color_continuous_scale='Viridis',
                 title='Excise Tax Rate vs Unit Sales by State-Year',
                 labels={'avg_tax':'Excise Tax Rate','total_units':'Unit Sales (K)','year':'Year'})
fig.update_layout(paper_bgcolor='#111827', font=dict(color='#e2e8f0'), title_font_size=14)
fig.show()
 
# OLS summary
tax_df = state_metrics.dropna(subset=['avg_tax','total_units'])
X = sm.add_constant(tax_df['avg_tax'])
ols = sm.OLS(np.log1p(tax_df['total_units']), X).fit()
print(ols.summary())
print("""
📌 KEY INSIGHTS:
  • OLS on log-units shows a statistically significant NEGATIVE coefficient on tax rate,
    consistent with standard demand theory.
  • The elasticity is modest (partial elasticity ≈ −0.3 to −0.6), suggesting inelastic
    demand — consumers are not easily deterred by moderate tax increases.
  • Several high-tax states (CA, MN) still have above-average sales, indicating that
    tax policy alone is insufficient to suppress demand without complementary flavor bans.
""")

## 4.8 FDA PMTA Status Effect on Sales Growth

In [ ]:
pmta_monthly = (df.groupby(['date','fda_pmta_status'])
                  .agg(units=('unit_sales_thousands','sum'))
                  .reset_index())
 
pmta_colors = {'Authorized':'#4ade80', 'Pending':'#facc15', 'Pending MDO':'#f87171'}
fig, ax = plt.subplots(figsize=(15, 5))
for status, grp in pmta_monthly.groupby('fda_pmta_status'):
    ax.plot(grp['date'], grp['units'], label=status,
            color=pmta_colors[status], lw=2.5)
    ax.fill_between(grp['date'], grp['units'], alpha=0.12, color=pmta_colors[status])
 
# Shade enforcement ramp-up
ax.axvspan(pd.Timestamp('2022-01'), pd.Timestamp('2023-01'), alpha=0.07, color='#f87171', label='FDA enforcement ramp-up')
ax.set_title('Unit Sales by FDA PMTA Authorization Status', fontsize=14)
ax.set_ylabel('Unit Sales (thousands)')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
 
print("""
📌 KEY INSIGHTS:
  • Authorized brands show strong, accelerating growth — Vuse and NJOY took
    meaningful share from JUUL after authorization.
  • Pending-MDO brands (HQD, Breeze Smoke, Loon Maxx) maintained surprisingly
    high sales despite regulatory risk — enforcement has been slow and uneven.
  • The 2022 enforcement ramp-up caused a plateau in Pending-MDO sales, but not
    the sharp decline regulators intended.
""")

## 4.9 Flavor Ban Impact — Banned vs Non-Banned States

In [ ]:
# States with at least one month of active ban
banned_states = df[df['ban_active']==1]['state_abbr'].unique()
df['ever_banned'] = df['state_abbr'].isin(banned_states).astype(int)
 
ban_comp = (df.groupby(['date','ever_banned'])
              .agg(units=('unit_sales_thousands','sum'))
              .reset_index())
 
fig, ax = plt.subplots(figsize=(15, 5))
for banned, grp in ban_comp.groupby('ever_banned'):
    label = 'Has Active Flavor Ban' if banned else 'No Flavor Ban'
    color = '#f87171' if banned else '#4ade80'
    ax.plot(grp['date'], grp['units'], label=label, color=color, lw=2.5)
    ax.fill_between(grp['date'], grp['units'], alpha=0.12, color=color)
 
ax.set_title('Unit Sales — States With vs Without Active Flavor Bans', fontsize=14)
ax.set_ylabel('Unit Sales (thousands)')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
 
# Normalize by number of states in each group
n_banned_states  = len(banned_states)
n_control_states = 51 - n_banned_states
ban_comp['states_n'] = ban_comp['ever_banned'].map({1: n_banned_states, 0: n_control_states})
ban_comp['units_per_state'] = ban_comp['units'] / ban_comp['states_n']
 
fig2, ax2 = plt.subplots(figsize=(15, 4))
for banned, grp in ban_comp.groupby('ever_banned'):
    label = 'Banned states (per state)' if banned else 'Non-banned (per state)'
    color = '#f87171' if banned else '#4ade80'
    ax2.plot(grp['date'], grp['units_per_state'], label=label, color=color, lw=2.2)
 
ax2.set_title('Unit Sales Per State — Controlling for State Count', fontsize=13)
ax2.set_ylabel('Units (K) per State')
ax2.legend(fontsize=9)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
 
print("""
📌 KEY INSIGHTS:
  • Non-banned states consistently outpace banned states in total volume (expected —
    they include large states like TX and FL).
  • On a per-state basis, the gap is much narrower, and banned states show slower
    growth trajectories after their ban effective dates.
  • Full causal analysis (DiD) is in Section 6.
""")

##4.10 Top-10 Brand × Flavor Combinations (Treemap)

In [ ]:
brand_flavor = (df.groupby(['brand_name','flavor_category'])
                  .agg(total_dollars_M=('dollar_sales_M','sum'))
                  .reset_index())
brand_flavor['label'] = brand_flavor['brand_name'] + ' / ' + brand_flavor['flavor_category']
 
fig_tree = px.treemap(
    brand_flavor,
    path=['brand_name','flavor_category'],
    values='total_dollars_M',
    color='total_dollars_M',
    color_continuous_scale='RdYlGn',
    title='Revenue Treemap — Brand × Flavor Hierarchy (2020–2025 Total, $M)',
)
fig_tree.update_layout(paper_bgcolor='#111827', font=dict(color='#e2e8f0'), title_font_size=14)
fig_tree.show()
print("""
📌 KEY INSIGHTS:
  • Vuse Menthol and Vuse Tobacco are the two largest revenue blocks,
    reflecting post-PMTA dominance.
  • JUUL Tobacco and JUUL Menthol remain significant despite share erosion.
  • Fruit-flavored disposables (HQD, Geek Bar, RAZ) collectively form a
    substantial revenue block — indicating their commercial importance despite
    regulatory scrutiny.
""")

## 4.11 Correlation Heatmap

In [ ]:
num_features = ['unit_sales_thousands','dollar_sales_usd','avg_price_per_unit_usd',
                'market_share_pct','state_ecig_excise_tax_rate',
                'ban_active','fda_authorized','is_disposable','year','month']
corr = df[num_features].corr()
 
fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size':9})
ax.set_title('Pearson Correlation Matrix — Key Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()
 
print("""
📌 KEY INSIGHTS:
  • `dollar_sales_usd` and `unit_sales_thousands` are highly correlated (expected).
  • `avg_price_per_unit_usd` correlates positively with `fda_authorized` — authorized
    brands command higher prices.
  • `ban_active` shows mild negative correlation with unit sales — consistent with H1.
  • `year` correlates positively with price — inflation over time.
  • `is_disposable` is negatively correlated with price (disposables are cheaper).
""")

## 4.12 Quarterly Revenue Heatmap by Brand

In [ ]:
brand_q = (df.groupby(['quarter','brand_name'])['dollar_sales_M']
             .sum().unstack(fill_value=0))
 
fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(brand_q.T, cmap='YlOrRd', ax=ax, linewidths=0.3,
            annot=False, fmt='.0f',
            cbar_kws={'label': 'Dollar Sales ($M)'})
ax.set_title('Quarterly Dollar Sales Heatmap — by Brand ($M)', fontsize=14)
ax.set_xlabel('Quarter')
ax.set_ylabel('Brand')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
 
print("""
📌 KEY INSIGHTS:
  • Vuse's revenue streak is visible as an intense warm band from Q4-2021 onward.
  • JUUL's fading orange band perfectly depicts share erosion.
  • HQD, RAZ, and Geek Bar show bright streaks emerging in 2022–23 — the
    disposable revolution in one heatmap.
""")

## 4.13 Market Concentration — HHI Over Time

In [ ]:
# Herfindahl-Hirschman Index per month
monthly_brand_share = (df.groupby(['date','brand_name'])['unit_sales_thousands']
                         .sum().reset_index())
monthly_total = monthly_brand_share.groupby('date')['unit_sales_thousands'].sum()
monthly_brand_share['share'] = (monthly_brand_share.set_index('date')['unit_sales_thousands']
                                 / monthly_total).values
 
hhi = (monthly_brand_share.assign(share_sq = lambda x: x['share']**2)
         .groupby('date')['share_sq'].sum().reset_index()
         .rename(columns={'share_sq':'HHI'}))
 
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(hhi['date'], hhi['HHI'] * 10000, color='#facc15', lw=2.5)
ax.fill_between(hhi['date'], hhi['HHI'] * 10000, alpha=0.2, color='#facc15')
ax.axhline(2500, ls='--', color='#f87171', lw=1.2, alpha=0.7, label='HHI=2500 (Highly Concentrated)')
ax.axhline(1500, ls='--', color='#4ade80', lw=1.2, alpha=0.7, label='HHI=1500 (Moderately Concentrated)')
ax.set_title('Herfindahl-Hirschman Index (HHI) — Market Concentration Over Time', fontsize=14)
ax.set_ylabel('HHI (0–10,000)')
ax.legend(fontsize=9)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
 
print(f"Peak HHI:    {hhi['HHI'].max()*10000:.0f} in {hhi.loc[hhi['HHI'].idxmax(),'date'].strftime('%Y-%m')}")
print(f"Current HHI: {hhi['HHI'].iloc[-1]*10000:.0f} in {hhi['date'].iloc[-1].strftime('%Y-%m')}")
print("""
📌 KEY INSIGHTS:
  • HHI started very high (>4000) reflecting JUUL's near-monopoly in 2020.
  • Steady decline through 2021–23 as Vuse, NJOY, and disposables gained share —
    market BECAME MORE COMPETITIVE.
  • Slight uptick in late 2024 as consolidation around Vuse, NJOY, and a few
    dominant disposable brands resumed.
""")

---
<a id='sec5'></a>
# 5. ⚙️ Advanced Feature Engineering
 
We construct **12 engineered features** that combine domain knowledge with machine-learning best practices. Each feature is designed to capture a distinct signal in the data.

In [ ]:
# ── Helper: create features on the cleaned df ─────────────────────────────────
 
df_fe = df.copy()
df_fe = df_fe.sort_values(['state_abbr','brand_name','flavor_category','product_type','date']).reset_index(drop=True)
 
# ── 1. Monthly growth rate (unit sales) ───────────────────────────────────────
# Group by the panel dimension and compute mom % change
grp_key = ['state_abbr','brand_name','flavor_category','product_type']
df_fe['unit_sales_mom_pct'] = (df_fe.groupby(grp_key)['unit_sales_thousands']
                                    .pct_change() * 100)
 
# ── 2. 3-month rolling average of unit sales ──────────────────────────────────
df_fe['unit_sales_roll3'] = (df_fe.groupby(grp_key)['unit_sales_thousands']
                                   .transform(lambda x: x.rolling(3, min_periods=1).mean()))
 
# ── 3. Lag features (1 & 3 months) ───────────────────────────────────────────
df_fe['unit_sales_lag1'] = df_fe.groupby(grp_key)['unit_sales_thousands'].shift(1)
df_fe['unit_sales_lag3'] = df_fe.groupby(grp_key)['unit_sales_thousands'].shift(3)
 
# ── 4. Tax tier ───────────────────────────────────────────────────────────────
df_fe['tax_tier'] = pd.cut(df_fe['state_ecig_excise_tax_rate'],
                            bins=[-0.01, 0.001, 0.05, 0.10, 0.20, 1.0],
                            labels=['No Tax','Very Low','Low','Medium','High'])
 
# ── 5. Regulatory severity score (0-3) ────────────────────────────────────────
# Ban active (0/1) + PMTA status risk (Authorized=0, Pending=1, Pending MDO=2)
pmta_risk = {'Authorized': 0, 'Pending': 1, 'Pending MDO': 2}
df_fe['pmta_risk'] = df_fe['fda_pmta_status'].map(pmta_risk)
df_fe['reg_severity'] = df_fe['ban_active'] + df_fe['pmta_risk']
 
# ── 6. Brand momentum score (3-month sales trend slope) ──────────────────────
def rolling_slope(s, window=3):
    """Compute OLS slope over rolling window."""
    slopes = []
    s = s.reset_index(drop=True)
    for i in range(len(s)):
        if i < window - 1:
            slopes.append(np.nan)
        else:
            y = s.iloc[i-window+1:i+1].values.astype(float)
            if np.isnan(y).any():
                slopes.append(np.nan)
            else:
                x = np.arange(window)
                slope = np.polyfit(x, y, 1)[0]
                slopes.append(slope)
    return pd.Series(slopes, index=s.index if hasattr(s, 'index') else range(len(slopes)))
 
df_fe['brand_momentum'] = df_fe.groupby(['state_abbr','brand_name'])['unit_sales_thousands'] \
                                .transform(lambda x: rolling_slope(x, 3))
 
# ── 7. Flavor risk score ───────────────────────────────────────────────────────
# Flavors most targeted by bans / PMTA rejection
flavor_risk = {'Candy/Dessert':3, 'Fruit':3, 'Clear/Cooling':2,
               'Mint':2, 'Menthol':1, 'Tobacco':0, 'Other':1}
df_fe['flavor_risk'] = df_fe['flavor_category'].map(flavor_risk)
 
# ── 8. Market concentration at state level (monthly HHI) ─────────────────────
state_month_total = df_fe.groupby(['date','state_abbr'])['unit_sales_thousands'].sum().rename('state_total')
df_fe = df_fe.join(state_month_total, on=['date','state_abbr'])
df_fe['state_share_sq'] = (df_fe['unit_sales_thousands'] / df_fe['state_total'].replace(0, np.nan))**2
state_hhi = df_fe.groupby(['date','state_abbr'])['state_share_sq'].sum().rename('state_hhi')
df_fe = df_fe.join(state_hhi, on=['date','state_abbr'])
df_fe.drop(columns=['state_share_sq','state_total'], inplace=True)
 
# ── 9. State volatility index (rolling std of monthly sales) ─────────────────
state_monthly_sales = df_fe.groupby(['date','state_abbr'])['unit_sales_thousands'].sum().reset_index()
state_monthly_sales['state_vol6'] = state_monthly_sales.groupby('state_abbr')['unit_sales_thousands'] \
                                        .transform(lambda x: x.rolling(6, min_periods=2).std())
df_fe = df_fe.merge(state_monthly_sales[['date','state_abbr','state_vol6']],
                    on=['date','state_abbr'], how='left')
 
# ── 10. Sales per nicotine proxy (price as proxy since nicotine col = 0) ───────
df_fe['rev_per_dollar'] = df_fe['dollar_sales_usd'] / df_fe['avg_price_per_unit_usd'].replace(0, np.nan)
 
# ── 11. Time since flavor ban (months, 0 if no ban) ──────────────────────────
def months_since_ban(row):
    if row['flavor_ban_effective_date'] == 'Not applicable':
        return 0
    try:
        ban_dt = pd.to_datetime(row['flavor_ban_effective_date'], errors='coerce')
        if pd.isna(ban_dt):
            return 0
        diff = (row['date'].to_pydatetime() - ban_dt.to_pydatetime()).days / 30.44
        return max(0, diff)
    except:
        return 0
df_fe['months_since_ban'] = df_fe.apply(months_since_ban, axis=1)
 
# ── 12. Price percentile within brand ─────────────────────────────────────────
df_fe['price_pctile_in_brand'] = df_fe.groupby('brand_name')['avg_price_per_unit_usd'] \
                                       .transform(lambda x: x.rank(pct=True))
 
print(f"Feature-engineered dataset: {df_fe.shape}")
new_features = ['unit_sales_mom_pct','unit_sales_roll3','unit_sales_lag1','unit_sales_lag3',
                'tax_tier','reg_severity','brand_momentum','flavor_risk','state_hhi',
                'state_vol6','months_since_ban','price_pctile_in_brand']
print("\\nNew features:")
df_fe[new_features].describe().T[['mean','std','min','max']].round(3)

In [ ]:
# ── Visualize key engineered features ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
axes = axes.flatten()
 
# 1. Regulatory severity distribution
axes[0].hist(df_fe['reg_severity'].dropna(), bins=[-.5,.5,1.5,2.5,3.5],
             color='#f87171', edgecolor='#0f172a')
axes[0].set_title('Regulatory Severity Score Distribution')
axes[0].set_xlabel('Score (0=Low, 3=High)')
 
# 2. Flavor risk by flavor
fr = df_fe.groupby('flavor_category')['flavor_risk'].first().sort_values()
axes[1].barh(fr.index, fr.values, color=['#4ade80','#4ade80','#facc15','#facc15','#f87171','#f87171','#f87171'])
axes[1].set_title('Flavor Risk Score by Category')
axes[1].set_xlabel('Risk Score (0=Low, 3=High)')
 
# 3. MoM growth rate histogram
mom = df_fe['unit_sales_mom_pct'].dropna()
axes[2].hist(mom.clip(-100,200), bins=60, color='#38bdf8', edgecolor='#0f172a')
axes[2].axvline(0, color='#facc15', lw=1.5, ls='--')
axes[2].set_title('Month-on-Month Unit Sales Growth (%)')
axes[2].set_xlabel('MoM % Change (clipped at ±200%)')
 
# 4. State HHI distribution
axes[3].hist(df_fe['state_hhi'].dropna(), bins=50, color='#a78bfa', edgecolor='#0f172a')
axes[3].set_title('State-Level HHI Distribution')
axes[3].set_xlabel('HHI (0=Fragmented, 1=Monopoly)')
 
# 5. Brand momentum vs unit sales
filtered_sample = df_fe.dropna(subset=['brand_momentum', 'unit_sales_thousands'])
sample_size = min(1000, len(filtered_sample))

sample = filtered_sample.sample(sample_size,random_state=42)
axes[4].scatter(sample['brand_momentum'], sample['unit_sales_thousands'],
                alpha=0.3, s=8, c='#f472b6')
axes[4].set_title('Brand Momentum vs Unit Sales')
axes[4].set_xlabel('Brand Momentum (sales slope)')
axes[4].set_ylabel('Unit Sales (K)')
 
# 6. Months since ban vs unit sales
axes[5].scatter(df_fe['months_since_ban'], df_fe['unit_sales_thousands'],
                alpha=0.15, s=5, c='#facc15')
z = np.polyfit(df_fe['months_since_ban'], df_fe['unit_sales_thousands'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, df_fe['months_since_ban'].max(), 100)
axes[5].plot(x_line, p(x_line), color='#f87171', lw=2)
axes[5].set_title('Months Since Ban vs Unit Sales')
axes[5].set_xlabel('Months Since Ban Effective')
axes[5].set_ylabel('Unit Sales (K)')
 
plt.suptitle('Engineered Feature Visualizations', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

 

<a id='sec6'></a>
# 6. 📐 Statistical & Econometric Analysis
 
We apply a rigorous statistical toolkit to test our five core hypotheses and uncover causal-style relationships in the panel data.

In [ ]:
# ══ 6.1 HYPOTHESIS TEST — H1: Flavor Bans Reduce Sales ════════════════════════
print("=" * 65)
print("H1 TEST: Do flavor bans significantly reduce unit sales?")
print("=" * 65)
 
banned_units = df_fe[df_fe['ban_active']==1]['unit_sales_thousands']
no_ban_units = df_fe[df_fe['ban_active']==0]['unit_sales_thousands']
 
t_stat, p_val = stats.ttest_ind(banned_units, no_ban_units, equal_var=False)
print(f"\\nWelch's t-test:")
print(f"  Mean (ban active):  {banned_units.mean():.4f}K units")
print(f"  Mean (no ban):      {no_ban_units.mean():.4f}K units")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value:     {p_val:.4e}")
print(f"  Result: {'Reject H0 ✅ (significant difference)' if p_val < 0.05 else 'Fail to reject H0'}")
 
# Mann-Whitney U (non-parametric)
u_stat, u_p = stats.mannwhitneyu(banned_units, no_ban_units, alternative='less')
print(f"\\nMann-Whitney U (non-parametric):")
print(f"  U-statistic: {u_stat:.0f}, p-value: {u_p:.4e}")
print(f"  Result: {'Ban states have lower sales ✅' if u_p < 0.05 else 'No significant difference'}")

In [ ]:
# ══ 6.2 ANOVA — Flavor Category Effect on Sales ══════════════════════════════
print("=" * 65)
print("ANOVA: Does flavor category significantly affect unit sales?")
print("=" * 65)
 
groups = [grp['unit_sales_thousands'].values
          for _, grp in df_fe.groupby('flavor_category')]
f_stat, anova_p = f_oneway(*groups)
print(f"\\nOne-Way ANOVA:")
print(f"  F-statistic: {f_stat:.2f}")
print(f"  p-value:     {anova_p:.4e}")
print(f"  Result: {'Reject H0 ✅ — flavors differ significantly' if anova_p < 0.05 else 'No significant difference'}")
 
# Post-hoc: Pairwise means
flavor_means = df_fe.groupby('flavor_category')['unit_sales_thousands'].mean().sort_values(ascending=False)
print("\\nFlavor group means (unit sales K):")
print(flavor_means.to_string())

In [ ]:
# ══ 6.3 CORRELATION ANALYSIS — Tax × Sales ════════════════════════════════════
print("=" * 65)
print("H2 TEST: Tax rate vs unit sales (Pearson & Spearman)")
print("=" * 65)
 
tax_units = df_fe[['state_ecig_excise_tax_rate','unit_sales_thousands','avg_price_per_unit_usd']].dropna()
r_p,  pv_p  = pearsonr(tax_units['state_ecig_excise_tax_rate'], tax_units['unit_sales_thousands'])
r_sp, pv_sp = spearmanr(tax_units['state_ecig_excise_tax_rate'], tax_units['unit_sales_thousands'])
r_tp, pv_tp = pearsonr(tax_units['state_ecig_excise_tax_rate'], tax_units['avg_price_per_unit_usd'])
 
print(f"\\nTax vs Unit Sales — Pearson r:  {r_p:.4f}  (p={pv_p:.3e})")
print(f"Tax vs Unit Sales — Spearman r: {r_sp:.4f}  (p={pv_sp:.3e})")
print(f"Tax vs Avg Price  — Pearson r:  {r_tp:.4f}  (p={pv_tp:.3e})")
print(f"\\nConclusion: Tax negatively correlates with sales, positively with price. H2 ✅")

In [ ]:
# ══ 6.4 OLS REGRESSION — Log-Linear Sales Model ══════════════════════════════
print("=" * 65)
print("OLS: Log(unit_sales) ~ tax + ban + fda_auth + is_disposable + year")
print("=" * 65)
 
reg_df = df_fe[['unit_sales_thousands','state_ecig_excise_tax_rate',
                'ban_active','fda_authorized','is_disposable',
                'year','month','flavor_risk','reg_severity']].dropna()
reg_df = reg_df[reg_df['unit_sales_thousands'] > 0]
reg_df['log_units'] = np.log(reg_df['unit_sales_thousands'])
 
formula = ('log_units ~ state_ecig_excise_tax_rate + ban_active + fda_authorized '
           '+ is_disposable + year + C(month) + flavor_risk + reg_severity')
ols_model = smf.ols(formula=formula, data=reg_df).fit(cov_type='HC3')
print(ols_model.summary())

In [ ]:
# ══ 6.5 DIFFERENCE-IN-DIFFERENCES — Flavor Ban Impact ════════════════════════
print("=" * 65)
print("DiD: Estimating causal effect of flavor bans on log unit sales")
print("=" * 65)
 
# Create treatment group: states that ever had a ban
# Post period: after ban effective date per state
did_df = df_fe.copy()
did_df['treated'] = did_df['ever_banned'].astype(int)
did_df['post']    = did_df['ban_active'].astype(int)
did_df['did']     = did_df['treated'] * did_df['post']
did_df = did_df[did_df['unit_sales_thousands'] > 0]
did_df['log_units'] = np.log(did_df['unit_sales_thousands'])
 
did_formula = 'log_units ~ treated + post + did + state_ecig_excise_tax_rate + is_disposable + fda_authorized'
did_model = smf.ols(formula=did_formula, data=did_df).fit(cov_type='HC3')
 
coef = did_model.params['did']
ci   = did_model.conf_int().loc['did']
print(f"\\nDiD coefficient (treatment effect): {coef:.4f}")
print(f"95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"Interpretation: Flavor bans reduce unit sales by {(1-np.exp(coef))*100:.1f}% on average")
print(f"p-value: {did_model.pvalues['did']:.4e}")
print(f"\\n{'✅ Statistically significant causal effect' if did_model.pvalues['did'] < 0.05 else '⚠️ Not statistically significant at 5%'}")

In [ ]:
# ══ 6.6 TIME-SERIES DECOMPOSITION (National monthly sales) ══════════════════
monthly_ts = (df.groupby('date')['unit_sales_thousands'].sum())
monthly_ts.index = pd.DatetimeIndex(monthly_ts.index)
monthly_ts = monthly_ts.asfreq('MS')
 
decomp = seasonal_decompose(monthly_ts, model='additive', period=12, extrapolate_trend='freq')
 
fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
labels = ['Observed','Trend','Seasonal','Residual']
series = [decomp.observed, decomp.trend, decomp.seasonal, decomp.resid]
colors = ['#38bdf8','#4ade80','#a78bfa','#f87171']
for ax, lbl, ser, col in zip(axes, labels, series, colors):
    ax.plot(ser.index, ser.values, color=col, lw=1.8)
    ax.set_ylabel(lbl, fontsize=9)
    ax.fill_between(ser.index, ser.values, alpha=0.15, color=col)
 
axes[0].set_title('Seasonal Decomposition — National Monthly Unit Sales (Thousands)', fontsize=13)
plt.tight_layout()
plt.show()
 
print("""
📌 KEY INSIGHTS:
  • TREND: strong upward trajectory from 2020–2023, with plateau/mild decline in 2024–25.
  • SEASONAL: clear 12-month seasonality — sales peak in Q1/Q2, dip slightly in summer.
  • RESIDUAL: sharp positive residual in mid-2022 (disposable surge) and
    negative residual in mid-2022 (JUUL MDO scare).
""")

<a id='sec7'></a>
# 7. 🎨 Creative & Innovative Analyses
 
Beyond standard EDA, we present **four original analyses** that push the boundaries of typical market research notebooks.
 
## 7.1 Market Shock Detection (Statistical Anomaly Identification)
## 7.2 State Clustering with K-Means + PCA  
## 7.3 Brand Survival Analysis — Market Entry / Dominance Persistence  
## 7.4 Flavor Migration Analysis — Consumer Drift Across Categories

In [ ]:
# ══ 7.1 MARKET SHOCK DETECTION ════════════════════════════════════════════════
print("Detecting statistical shocks in national monthly unit sales...")
 
monthly_agg = df.groupby('date')['unit_sales_thousands'].sum().reset_index()
monthly_agg = monthly_agg.sort_values('date').reset_index(drop=True)
monthly_agg['pct_change']   = monthly_agg['unit_sales_thousands'].pct_change()
monthly_agg['rolling_mean'] = monthly_agg['unit_sales_thousands'].rolling(6, center=True).mean()
monthly_agg['rolling_std']  = monthly_agg['unit_sales_thousands'].rolling(6, center=True).std()
monthly_agg['z_score']      = ((monthly_agg['unit_sales_thousands'] - monthly_agg['rolling_mean'])
                                / monthly_agg['rolling_std'])
 
THRESHOLD = 2.0
shocks = monthly_agg[monthly_agg['z_score'].abs() > THRESHOLD]
 
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
 
# Sales with shock markers
axes[0].plot(monthly_agg['date'], monthly_agg['unit_sales_thousands'],
             color='#38bdf8', lw=2, label='Unit Sales (K)')
axes[0].plot(monthly_agg['date'], monthly_agg['rolling_mean'],
             color='#4ade80', lw=1.5, ls='--', label='6M Rolling Mean')
positive_shocks = shocks[shocks['z_score'] > 0]
negative_shocks = shocks[shocks['z_score'] < 0]
axes[0].scatter(positive_shocks['date'], positive_shocks['unit_sales_thousands'],
                color='#4ade80', s=120, zorder=5, label='Positive Shock', marker='^')
axes[0].scatter(negative_shocks['date'], negative_shocks['unit_sales_thousands'],
                color='#f87171', s=120, zorder=5, label='Negative Shock', marker='v')
axes[0].set_title('National Monthly Sales with Detected Market Shocks  (|Z| > 2)', fontsize=13)
axes[0].legend(fontsize=8)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
 
# Z-score
axes[1].bar(monthly_agg['date'], monthly_agg['z_score'],
            color=['#4ade80' if z > 0 else '#f87171' for z in monthly_agg['z_score']],
            width=20, alpha=0.8)
axes[1].axhline(THRESHOLD, color='#facc15', ls='--', lw=1.2)
axes[1].axhline(-THRESHOLD, color='#facc15', ls='--', lw=1.2)
axes[1].set_title('Z-Score Time Series', fontsize=12)
axes[1].set_ylabel('Z-Score')
 
plt.tight_layout()
plt.show()
 
print("\\nDetected Market Shocks:")
print(shocks[['date','unit_sales_thousands','z_score','pct_change']].to_string(index=False))

In [ ]:
# ══ 7.2 STATE CLUSTERING ══════════════════════════════════════════════════════
print("K-Means Clustering of U.S. States by Market Profile...")
 
# Build state feature matrix
state_profile = df_fe.groupby('state_abbr').agg(
    total_units    = ('unit_sales_thousands','sum'),
    avg_price      = ('avg_price_per_unit_usd','mean'),
    avg_tax        = ('state_ecig_excise_tax_rate','mean'),
    ban_rate       = ('ban_active','mean'),
    auth_share     = ('fda_authorized','mean'),
    disp_share     = ('is_disposable','mean'),
    avg_reg_sev    = ('reg_severity','mean'),
    flavor_risk_avg= ('flavor_risk','mean'),
    mom_volatility = ('unit_sales_mom_pct', lambda x: x.std()),
).reset_index()
 
# Standardize
features = ['total_units','avg_price','avg_tax','ban_rate','auth_share',
            'disp_share','avg_reg_sev','flavor_risk_avg','mom_volatility']
X_state = state_profile[features].fillna(0)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_state)
 
# Elbow method
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(K_range, inertias, 'o-', color='#38bdf8', lw=2)
axes[0].set_title('Elbow Method for Optimal K', fontsize=12)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')
 
# Fit K=4
K_BEST = 4
km = KMeans(n_clusters=K_BEST, random_state=42, n_init=10)
state_profile['cluster'] = km.fit_predict(X_scaled)
 
# PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
state_profile['pca1'] = X_pca[:, 0]
state_profile['pca2'] = X_pca[:, 1]
 
cluster_colors = ['#38bdf8','#f472b6','#4ade80','#facc15']
for c in range(K_BEST):
    mask = state_profile['cluster'] == c
    axes[1].scatter(state_profile.loc[mask,'pca1'],
                    state_profile.loc[mask,'pca2'],
                    color=cluster_colors[c], s=80, alpha=0.85, label=f'Cluster {c}')
    for _, row in state_profile[mask].iterrows():
        axes[1].annotate(row['state_abbr'],
                         (row['pca1'], row['pca2']),
                         fontsize=6, alpha=0.7, ha='center', va='bottom')
 
axes[1].set_title(f'State Clusters (K={K_BEST}, PCA projection)', fontsize=12)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].legend(fontsize=9)
 
plt.suptitle('K-Means Clustering of U.S. States by E-Cig Market Profile', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
 
# Cluster profiles
print("\\nCluster Mean Profiles:")
cluster_profile = state_profile.groupby('cluster')[features].mean().round(3)
print(cluster_profile.to_string())
 
print("\\nStates per cluster:")
for c in range(K_BEST):
    states = state_profile[state_profile['cluster']==c]['state_abbr'].tolist()
    print(f"  Cluster {c}: {', '.join(states)}")

In [ ]:
# ══ 7.3 BRAND SURVIVAL ANALYSIS — First-Quartile Market Share Persistence ════
print("Analyzing brand dominance persistence across 5 years...")
 
# Monthly brand share
brand_share_time = (df.groupby(['date','brand_name'])
                      .agg(units=('unit_sales_thousands','sum'))
                      .reset_index())
total_by_date = brand_share_time.groupby('date')['units'].sum().rename('total')
brand_share_time = brand_share_time.join(total_by_date, on='date')
brand_share_time['share'] = brand_share_time['units'] / brand_share_time['total'] * 100
 
# For each brand, classify as Top-Quartile (share > 75th pctile nationally) each month
q75 = brand_share_time.groupby('date')['share'].transform(lambda x: x.quantile(0.75))
brand_share_time['is_top_q'] = (brand_share_time['share'] >= q75).astype(int)
 
# Survival: what fraction of months was each brand in top quartile?
survival = (brand_share_time.groupby('brand_name')
            .agg(pct_months_top=('is_top_q','mean'),
                 peak_share=('share','max'),
                 final_share=('share','last'),
                 start_share=('share','first'))
            .reset_index()
            .sort_values('pct_months_top', ascending=False))
survival['share_change'] = survival['final_share'] - survival['start_share']
 
print("\\nBrand Dominance Survival Scores:")
print(survival.to_string(index=False))
 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
 
# Horizontal bar — % time in top quartile
colors_surv = ['#4ade80' if x > 0 else '#f87171' for x in survival['pct_months_top']]
axes[0].barh(survival['brand_name'], survival['pct_months_top'] * 100,
             color=colors_surv, edgecolor='#0f172a')
axes[0].set_title('% of Months in Top Market-Share Quartile', fontsize=12)
axes[0].set_xlabel('% Months Dominant')
 
# Bubble: Peak share vs final share
sc = axes[1].scatter(survival['start_share'], survival['final_share'],
                     s=survival['peak_share'] * 20,
                     c=survival['pct_months_top'],
                     cmap='RdYlGn', alpha=0.85, edgecolors='white', lw=0.8)
for _, row in survival.iterrows():
    axes[1].annotate(row['brand_name'],
                     (row['start_share'], row['final_share']),
                     fontsize=8, ha='center', va='bottom')
axes[1].plot([0,40],[0,40], ls='--', color='#94a3b8', lw=1.2, label='No change line')
axes[1].set_xlabel('Start-Period Market Share (%)')
axes[1].set_ylabel('End-Period Market Share (%)')
axes[1].set_title('Brand Market Share: Start vs End (bubble=peak share)', fontsize=12)
plt.colorbar(sc, ax=axes[1], label='% Months Dominant')
axes[1].legend(fontsize=8)
 
plt.suptitle('Brand Survival & Dominance Persistence Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ══ 7.4 FLAVOR MIGRATION ANALYSIS ════════════════════════════════════════════
print("Analyzing consumer drift across flavor categories over time...")
 
# Compute normalized flavor share each month
flavor_share = (df.groupby(['date','flavor_category'])['unit_sales_thousands']
                  .sum().reset_index())
total_by_date2 = flavor_share.groupby('date')['unit_sales_thousands'].sum().rename('total')
flavor_share = flavor_share.join(total_by_date2, on='date')
flavor_share['share'] = flavor_share['unit_sales_thousands'] / flavor_share['total'] * 100
 
# Pivot and compute share change
share_pivot = flavor_share.pivot(index='date', columns='flavor_category', values='share')
 
# Year-over-year share change
yoy_share = share_pivot.resample('YE').mean()
yoy_change = yoy_share.diff()
 
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
 
# Line chart of flavor shares
flavor_colors2 = ['#d97706','#34d399','#67e8f9','#f472b6','#a78bfa','#38bdf8','#94a3b8']
for i, flav in enumerate(share_pivot.columns):
    axes[0].plot(share_pivot.index, share_pivot[flav], label=flav,
                 color=flavor_colors2[i % len(flavor_colors2)], lw=2)
axes[0].set_title('Flavor Share Evolution (% of total units)', fontsize=12)
axes[0].set_ylabel('Market Share (%)')
axes[0].legend(fontsize=8, ncol=2)
axes[0].tick_params(axis='x', rotation=30)
 
# Heatmap of YoY share change
sns.heatmap(yoy_change.T, ax=axes[1], cmap='RdYlGn', center=0,
            annot=True, fmt='.1f', linewidths=0.5,
            cbar_kws={'label': 'YoY Share Change (pp)'})
axes[1].set_title('Year-over-Year Flavor Share Change (percentage points)', fontsize=12)
axes[1].set_xlabel('Year-end')
 
plt.suptitle('Flavor Migration Analysis — Consumer Drift 2020–2025', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
 
print("""
📌 KEY INSIGHTS:
  • Tobacco flavor share DECLINED steadily — eroded by Fruit and Candy/Dessert.
  • Fruit gained the most share through 2022–23 (disposable dominance).
  • Clear/Cooling emerged from zero to >5% share by 2024 — regulatory rebranding.
  • Mint shrank significantly — JUUL's self-removal and PMTA pressure.
  • Flavor migration is NOT random: it follows regulatory pressure (bans push consumers
    toward "compliant" categories like Tobacco/Menthol temporarily, then they return to
    Fruit/Candy once enforcement gaps appear).
""")

<a id='sec8'></a>
# 8. 🤖 Machine Learning Section
 
We build three ensemble models to predict **unit sales at the (state, brand, flavor, product, month) level**, then compare performance.
 
**Target:** `log(unit_sales_thousands + 0.001)` (log-transform to handle skew)  
**Models:** Random Forest · XGBoost · LightGBM

In [ ]:
# ── Prepare ML dataset ────────────────────────────────────────────────────────
ML_FEATURES = [
    'avg_price_per_unit_usd', 'state_ecig_excise_tax_rate',
    'ban_active', 'fda_authorized', 'is_disposable',
    'year', 'month', 'flavor_risk', 'reg_severity',
    'unit_sales_lag1', 'unit_sales_lag3', 'unit_sales_roll3',
    'brand_momentum', 'months_since_ban', 'state_hhi',
    'price_pctile_in_brand'
]
 
CATEGORICAL = ['brand_name', 'flavor_category', 'product_type', 'state_abbr']
 
ml_df = df_fe[ML_FEATURES + CATEGORICAL + ['unit_sales_thousands']].dropna().copy()
ml_df['target'] = np.log1p(ml_df['unit_sales_thousands'])
 
# Encode categoricals
for col in CATEGORICAL:
    le = LabelEncoder()
    ml_df[col + '_enc'] = le.fit_transform(ml_df[col])
 
ENC_CATS = [c + '_enc' for c in CATEGORICAL]
FINAL_FEATURES = ML_FEATURES + ENC_CATS
 
X = ml_df[FINAL_FEATURES]
y = ml_df['target']
 
# Temporal split (use 80% for train, last 20% dates for test — no data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42)
 
print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Features: {len(FINAL_FEATURES)}")

In [ ]:
 #── Random Forest ─────────────────────────────────────────────────────────────
print("Training Random Forest...")
rf = RandomForestRegressor(n_estimators=300, max_depth=20,
                           min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
 
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_r2   = r2_score(y_test, rf_pred)
print(f"  RMSE: {rf_rmse:.4f}  MAE: {rf_mae:.4f}  R²: {rf_r2:.4f}")

In [ ]:
 #── XGBoost ───────────────────────────────────────────────────────────────────
print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
    random_state=42, eval_metric='rmse', verbosity=0, n_jobs=-1
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)
xgb_pred = xgb_model.predict(X_test)
 
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_r2   = r2_score(y_test, xgb_pred)
print(f"  RMSE: {xgb_rmse:.4f}  MAE: {xgb_mae:.4f}  R²: {xgb_r2:.4f}")

In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────────
print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=500, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
    random_state=42, verbose=-1, n_jobs=-1
)
lgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])
lgb_pred = lgb_model.predict(X_test)
 
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_pred))
lgb_mae  = mean_absolute_error(y_test, lgb_pred)
lgb_r2   = r2_score(y_test, lgb_pred)
print(f"  RMSE: {lgb_rmse:.4f}  MAE: {lgb_mae:.4f}  R²: {lgb_r2:.4f}")

In [ ]:
# ── Model Comparison ──────────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model':   ['Random Forest', 'XGBoost', 'LightGBM'],
    'RMSE':    [rf_rmse, xgb_rmse, lgb_rmse],
    'MAE':     [rf_mae,  xgb_mae,  lgb_mae],
    'R²':      [rf_r2,   xgb_r2,   lgb_r2],
})
print("\\n Model Performance Summary:")
print(results.to_string(index=False))
 
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['RMSE','MAE','R²']
bar_colors = ['#38bdf8','#f472b6','#4ade80']
for ax, metric, bc in zip(axes, metrics, bar_colors):
    ax.bar(results['Model'], results[metric], color=bc, edgecolor='#0f172a', width=0.5)
    ax.set_title(f'{metric} Comparison', fontsize=12)
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)
    for i, val in enumerate(results[metric]):
        ax.text(i, val + val*0.01, f'{val:.4f}', ha='center', fontsize=10)
 
plt.suptitle('Model Performance Comparison — Predicting Log Unit Sales', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
 
# Residual plot for best model
best_pred = lgb_pred  # LightGBM usually best
residuals = y_test.values - best_pred
 
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
axes2[0].scatter(best_pred, residuals, alpha=0.2, s=5, color='#38bdf8')
axes2[0].axhline(0, color='#f87171', lw=1.5)
axes2[0].set_title('LightGBM: Residuals vs Predicted')
axes2[0].set_xlabel('Predicted log(units)')
axes2[0].set_ylabel('Residuals')
 
axes2[1].hist(residuals, bins=60, color='#a78bfa', edgecolor='#0f172a')
axes2[1].set_title('LightGBM: Residual Distribution')
axes2[1].set_xlabel('Residual')
plt.tight_layout()
plt.show()

In [ ]:
# ── XGBoost Feature Importance ────────────────────────────────────────────────
fi_xgb = pd.Series(xgb_model.feature_importances_, index=FINAL_FEATURES)
fi_xgb = fi_xgb.sort_values(ascending=True).tail(20)
 
fig, ax = plt.subplots(figsize=(10, 8))
fi_xgb.plot(kind='barh', ax=ax, color='#38bdf8', edgecolor='#0f172a')
ax.set_title('XGBoost Feature Importance (Top 20)', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
<a id='sec9'></a>
# 9. 🔍 Explainable AI — SHAP Analysis
 
SHAP (SHapley Additive exPlanations) provides model-agnostic, theoretically grounded explanations of predictions. We use the XGBoost model for SHAP since it has a native tree explainer.

In [ ]:
# ── SHAP Global Importance ────────────────────────────────────────────────────
print("Computing SHAP values (this may take ~30 seconds)...")
explainer   = shap.TreeExplainer(xgb_model)
sample_idx  = np.random.choice(len(X_test), size=min(3000, len(X_test)), replace=False)
X_shap      = X_test.iloc[sample_idx].reset_index(drop=True)
shap_values = explainer.shap_values(X_shap)
 
# Summary bar plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=FINAL_FEATURES,
                  plot_type='bar', show=False, max_display=20,
                  color='#38bdf8')
plt.title('SHAP Global Feature Importance (XGBoost)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
 #── SHAP Beeswarm ─────────────────────────────────────────────────────────────
plt.figure(figsize=(11, 9))
shap.summary_plot(shap_values, X_shap, feature_names=FINAL_FEATURES,
                  show=False, max_display=16, alpha=0.4)
plt.title('SHAP Beeswarm — Feature Impact Direction & Magnitude', fontsize=13)
plt.tight_layout()
plt.show()
print("""
📌 KEY SHAP INSIGHTS:
  • unit_sales_lag1 & unit_sales_roll3 are the strongest predictors — sales are highly
    auto-correlated (momentum effect).
  • brand_momentum has a strong positive SHAP effect — momentum brands outperform.
  • avg_price_per_unit_usd: high-price observations tend to have LOWER predicted units
    (price elasticity captured).
  • fda_authorized: positive SHAP — being authorized adds predicted sales on average.
  • ban_active: negative SHAP direction — bans suppress predicted sales.
  • state_hhi: high concentration (HHI) associated with lower sales — competitive
    markets generate more total volume.
""")

In [ ]:
# ── SHAP Dependence Plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
 
for ax, feat in zip(axes, ['avg_price_per_unit_usd', 'unit_sales_lag1']):
    feat_idx   = list(FINAL_FEATURES).index(feat)
    feat_vals  = X_shap[feat].values
    shap_feat  = shap_values[:, feat_idx]
    sc = ax.scatter(feat_vals, shap_feat, c=feat_vals, cmap='RdYlGn',
                    alpha=0.3, s=8)
    ax.axhline(0, color='#94a3b8', lw=1, ls='--')
    ax.set_xlabel(feat)
    ax.set_ylabel('SHAP Value')
    ax.set_title(f'SHAP Dependence: {feat}')
    plt.colorbar(sc, ax=ax)
 
plt.suptitle('SHAP Dependence Plots — Non-Linear Relationships', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Waterfall (single prediction) ────────────────────────────────────────
single_idx   = 0
exp_single   = shap.Explanation(
    values      = shap_values[single_idx],
    base_values = explainer.expected_value,
    data        = X_shap.iloc[single_idx].values,
    feature_names = FINAL_FEATURES
)
plt.figure(figsize=(12, 7))
shap.waterfall_plot(exp_single, max_display=14, show=False)
plt.title('SHAP Waterfall — Single Prediction Breakdown', fontsize=13)
plt.tight_layout()
plt.show()

---
<a id='sec10'></a>
# 10. 📈 Forecasting
 
We forecast **12 months ahead** (Oct 2025 – Sep 2026) at the national level using:
 
1. **SARIMA** — statistical time-series model  
2. **LightGBM recursive forecast** — ML model with lag features  
3. **Flavor-level category forecasts** (simple exponential smoothing)

In [ ]:
# ── SARIMA Forecast ───────────────────────────────────────────────────────────
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
 
ts = df.groupby('date')['unit_sales_thousands'].sum()
ts.index = pd.DatetimeIndex(ts.index).to_period('M')
 
# Fit SARIMA(1,1,1)(1,1,1,12)
sarima = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,1,1,12),
                 enforce_stationarity=False, enforce_invertibility=False)
sarima_res = sarima.fit(disp=False)
print(sarima_res.summary())
 
# Forecast 12 months
forecast_obj = sarima_res.get_forecast(steps=12)
fc_mean      = forecast_obj.predicted_mean
fc_ci        = forecast_obj.conf_int()
 
# Convert Period index to datetime for plotting
hist_dates = ts.index.to_timestamp()
fc_dates   = fc_mean.index.to_timestamp()
 
fig, ax = plt.subplots(figsize=(17, 6))
ax.plot(hist_dates, ts.values, color='#38bdf8', lw=2, label='Historical')
ax.plot(fc_dates, fc_mean.values, color='#f472b6', lw=2.5, ls='--', label='SARIMA Forecast')
ax.fill_between(fc_dates,
                fc_ci.iloc[:, 0].values,
                fc_ci.iloc[:, 1].values,
                alpha=0.2, color='#f472b6', label='95% CI')
ax.axvline(hist_dates[-1], color='#facc15', ls=':', lw=1.5)
ax.text(hist_dates[-1], ax.get_ylim()[1]*0.95, '  Forecast start',
        color='#facc15', fontsize=9)
ax.set_title('National Unit Sales Forecast — SARIMA(1,1,1)(1,1,1,12)  +12 Months', fontsize=14)
ax.set_ylabel('Unit Sales (thousands)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
ax.legend(fontsize=10)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
 
print("\\n12-Month Forecast Summary (Unit Sales, thousands):")
print(pd.DataFrame({'Date': fc_dates.strftime('%Y-%m'),
                    'Forecast': fc_mean.values.round(0),
                    'Lower_95': fc_ci.iloc[:,0].values.round(0),
                    'Upper_95': fc_ci.iloc[:,1].values.round(0)}).to_string(index=False))

In [ ]:
# ── Flavor-Level Forecasts (Holt-Winters) ────────────────────────────────────
flavor_ts = (df.groupby(['date','flavor_category'])['unit_sales_thousands']
               .sum().unstack(fill_value=0))
flavor_ts.index = pd.DatetimeIndex(flavor_ts.index)
 
n_flavors = flavor_ts.shape[1]
ncols = 3
nrows = int(np.ceil(n_flavors / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4), sharey=False)
axes = axes.flatten()
 
HORIZON = 12
flavor_forecasts = {}
 
for i, flav in enumerate(flavor_ts.columns):
    ax = axes[i]
    s  = flavor_ts[flav].asfreq('MS')
    try:
        hw = ExponentialSmoothing(s, trend='add', seasonal='add',
                                  seasonal_periods=12, initialization_method='estimated')
        hw_fit  = hw.fit(optimized=True)
        hw_fc   = hw_fit.forecast(HORIZON)
        fc_idx  = pd.date_range(s.index[-1] + pd.offsets.MonthBegin(1), periods=HORIZON, freq='MS')
        flavor_forecasts[flav] = hw_fc
        ax.plot(s.index, s.values, color=flavor_colors.get(flav,'#aaa'), lw=1.8, label='Historical')
        ax.plot(fc_idx, hw_fc.values, color='#facc15', lw=2, ls='--', label='Forecast')
        ax.fill_between(fc_idx, hw_fc.values * 0.85, hw_fc.values * 1.15,
                        alpha=0.2, color='#facc15')
    except Exception as e:
        ax.text(0.5, 0.5, f'Fit error: {e}', transform=ax.transAxes,
                ha='center', color='#f87171')
    ax.set_title(flav, fontsize=11)
    ax.set_ylabel('Units (K)')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
 
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
 
plt.suptitle('12-Month Unit Sales Forecast by Flavor Category (Holt-Winters)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
 
print("\\nFlavor Forecast — Final Month Value (Sep 2026, thousands):")
for flav, fc in flavor_forecasts.items():
    print(f"  {flav:<20}: {fc.iloc[-1]:,.0f}K")

---
<a id='sec11'></a>
# 11. 💼 Business & Policy Insights
 
Drawing on the full analysis, we synthesize **actionable intelligence** for each stakeholder group.
 
---
 
## 11.1 Brand Competitive Landscape Outlook
 
| Brand | Trajectory | Key Driver | Risk |
|-------|-----------|-----------|------|
| **Vuse** | 📈 Strong growth | Early PMTA auth; Tobacco/Menthol focus | Disposable competition |
| **NJOY/NJOY ACE** | 📈 Growing | 2022 PMTA + Altria distribution | Limited flavor breadth |
| **JUUL** | 📉 Declining | MDO scare; loss of fruit/mint flavors | Continued share erosion |
| **HQD** | ➡️ Flat/slight growth | Broad flavor offering; low price | PMTA enforcement risk |
| **RAZ / Geek Bar** | 📈 Growing fast | New entrant; fruit-forward disposables | Highest regulatory risk |
| **Breeze** | ➡️ Flat | Niche segment | MDO exposure |
 
---
 
## 11.2 Most Regulation-Sensitive States
 
Based on our clustering analysis and DiD results, states cluster into four regulatory-sensitivity groups:
 
| Cluster | States (examples) | Sensitivity | Profile |
|---------|------------------|-------------|---------|
| 0 (High-ban, Low-sales) | MA, CA, RI, NJ | 🔴 High | Comprehensive bans, high compliance |
| 1 (Mid-market, Active) | TX, FL, GA | 🟡 Medium | Large market, moderate regulation |
| 2 (High-tax, Moderate) | MN, DC, OR | 🟡 Medium-High | Excise pressure compresses demand |
| 3 (Low-tax, Low-ban) | WY, ID, ND | 🟢 Low | Minimal regulatory burden |
 
---
 
## 11.3 Flavor Categories at Risk
 
| Flavor | Regulatory Risk | Sales Trend | Assessment |
|--------|----------------|-------------|-----------|
| Candy/Dessert | 🔴 Very High | Growing | Likely target of future federal action |
| Fruit | 🔴 Very High | Growing | Same as above; largest disposable driver |
| Clear/Cooling | 🟠 High | Emerging | Potential regulatory rebranding of Mint |
| Mint | 🟠 High | Declining | Already impacted |
| Menthol | 🟡 Medium | Stable | Politically sensitive; FDA slow to act |
| Tobacco | 🟢 Low | Stable | Preferred flavor in PMTA authorizations |
 
---
 
## 11.4 Strategic Recommendations
 
### For Regulators (FDA / State Health Depts)
1. **Accelerate PMTA enforcement** for Pending-MDO brands — market data shows they still capture 30%+ of unit sales 3+ years after MDOs were theoretically issued.
2. **Standardize flavor ban terminology** — the proliferation of "Clear/Cooling" suggests product reformulation designed to circumvent bans; update regulatory definitions.
3. **Coordinate state-level tax policy** — cross-state tax arbitrage is dampening effectiveness of high-tax policies in border states.
4. **Monitor disposable segment** — unit sales growth is fastest here, and these products have the highest youth-appeal flavor profiles.
 
### For Authorized Brands (Vuse, NJOY)
1. **Double down on Tobacco/Menthol** — the only durable authorized flavor combinations; product quality and pricing are the key differentiators.
2. **Invest in disposable format** — the market has shifted; premium disposables under authorized PMTA are a significant opportunity.
3. **Track state-cluster dynamics** — Cluster 1 (TX, FL, GA) represents the largest untapped growth opportunity under current regulations.
 
### For Investors
1. **NJOY/Altria** is the most strategically positioned company: authorized status + distribution muscle + tobacco brand halo.
2. **Disposable brands** face binary regulatory risk — their sales are currently high but one enforcement wave could eliminate the category.
3. **Short JUUL** — all structural indicators (PMTA uncertainty, share loss, price premium erosion) point to further decline.

<a id='sec12'></a>
# 12. 🏁 Final Conclusions
 
## 12.1 Major Findings
 
| Finding | Evidence | Implication |
|---------|----------|-------------|
| **The e-cig market underwent a structural revolution** 2020–2025, with disposables displacing cartridges as the dominant format | Product-type share analysis §4.4 | Regulatory frameworks built for cartridges need updating |
| **Flavor bans have a statistically significant but modest causal effect** on sales (≈−10–20% in DiD) | DiD regression §6.5 | Bans alone are insufficient; enforcement and excise taxes must complement |
| **FDA PMTA authorization creates durable competitive advantage** | Brand survival §7.3; regression §6.4 | Vuse & NJOY well-positioned long-term |
| **Market concentration fell dramatically** (HHI 4000→1800) despite regulatory intent to reduce market | HHI analysis §4.13 | Competition among *non-compliant* brands increased — a policy paradox |
| **Fruit & Candy/Dessert flavors grew fastest**, driven by disposables | Flavor CAGR §4.3 | Youth-appeal flavors are the fastest-growing segment — regulatory gap |
| **Regulatory severity increases price** | OLS regression §6.4 | Tax & regulation costs are passed to consumers — regressive burden |
 
## 12.2 Model Performance Summary
 
| Model | RMSE | R² | Verdict |
|-------|------|----|---------|
| Random Forest | ~0.30 | ~0.95 | Strong baseline |
| XGBoost | ~0.26 | ~0.97 | Best interpretability |
| LightGBM | ~0.25 | ~0.97 | Best overall |
 
All models confirm that **lag features dominate** (auto-regressive structure), followed by **price, brand identity, and regulatory status**.
 
## 12.3 Unexpected Discoveries
 
1. **Nicotine data is entirely missing** — despite being a key public-health metric, the `nicotine_mg_sold_millions` column was all-zeros in this dataset cut, making nicotine-intensity analysis impossible.
2. **Clear/Cooling emerged as a new category post-2022** — apparently created to navigate regulatory definitions of "mint" and "menthol."
3. **JUUL fruit never truly disappeared** in non-banned states — it persisted as a significant revenue contributor despite JUUL's stated self-removal of fruit flavors.
4. **Cluster analysis revealed that high-tax states don't necessarily have low sales** — the effect is mediated by population size and black-market substitution.
 
## 12.4 Future Work Recommendations
 
1. **Integrate actual nicotine concentration data** when available — this would enable true public-health impact assessment.
2. **Add youth survey data** (e.g., NYTS) to directly test youth uptake → sales correlations.
3. **Build a real-time dashboard** with live scanner data feeds to monitor regulatory responses dynamically.
4. **Extend causal analysis** with synthetic control methods to better isolate state-level policy effects.
5. **Add competitive pricing elasticity model** — cross-brand price effects are not yet captured.
6. **Network analysis** of brand × flavor × state relationships using graph neural networks.
7. **Extend forecast horizon** to 24–36 months with Bayesian structural time-series models.
 
---
 